# Training CNNs for Remote Sensing Classification

This part of the final assignment follows Lab 5.1.

NOTE: This follows Lab 5.1 assuming we did not do the additions that we did do for checkpoint 2. Therefore, we might want to remove some parts of that code (we can do that when we have structured the code better).

In [7]:
# TODO: Remove not used imports and code

# Import required libraries
import os

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import lightning as pl
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.utils.data import WeightedRandomSampler

### Custom Dataset Class

Create a PyTorch Dataset for loading remote sensing data.

In [2]:
class YourCustomDataset(Dataset):
    """
    Custom Dataset for remote sensing data.

    Parameters:
    -----------
    data : numpy.ndarray
        Feature data (n_samples, n_features)
    labels : numpy.ndarray
        Target labels (n_samples,)
    """

    def __init__(self, data, labels):
        self.data = data
        self.labels = labels

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x = self.data[idx]
        y = self.labels[idx]
        return x, y

### Data Loading and Preprocessing

Load the training and validation datasets from CSV files.

In [4]:
# --- 1) Load from your .npz ---
npz_path = "/p/scratch/training2600/team1/data/T33VWH/training_data/combined_training_data.npz"
data = np.load(npz_path)

print("Keys in npz:", list(data.keys()))

if "patches" in data and "labels" in data:
    X = data["patches"]
    y = data["labels"]
else:
    possible_X_keys = ["X", "acquisitions", "inputs", "features", "patch"]
    possible_y_keys = ["y", "targets", "labels", "corine", "target"]

    X_key = next((k for k in possible_X_keys if k in data), None)
    y_key = next((k for k in possible_y_keys if k in data), None)

    if X_key is None or y_key is None:
        raise KeyError(
            f"Could not find (patches, labels) in {npz_path}. "
            f"Available keys: {list(data.keys())}"
        )

    X = data[X_key]
    y = data[y_key]

print(f"Loaded X shape: {X.shape}, dtype={X.dtype}")
print(f"Loaded y shape: {y.shape}, dtype={y.dtype}")

# --- 2) Preprocess ---
X = X.astype(np.float32) * 0.0001
X_flat = X.reshape(X.shape[0], -1)
print(f"Flattened X shape: {X_flat.shape}")
n_features = X_flat.shape[1]  # e.g. 36 for 3x3x4, or 90 for 3x3x10

# --- 3) Remap labels to contiguous 0..K-1 ---
# CORINE codes are sparse (1, 2, 3, 11, 12, ..., 25).
# Without remapping, num_classes = 26 but only 10 are real,
# leaving 16 ghost output neurons that confuse training.
unique_labels = np.unique(y)
label_to_idx = {int(lbl): i for i, lbl in enumerate(unique_labels)}
idx_to_label = {i: int(lbl) for i, lbl in enumerate(unique_labels)}
y = np.array([label_to_idx[int(lbl)] for lbl in y], dtype=np.int64)

print(f"\nLabel remapping (original -> index):")
for orig, idx in label_to_idx.items():
    print(f"  CORINE {orig:2d} -> class index {idx}")
print(f"Total classes: {len(unique_labels)}")


# --- 4) Split into train / val / test ---
def make_splits_train_val_test(
    X,
    y,
    train_frac=0.7,
    val_frac=0.15,
    test_frac=0.15,
    seed=42,
    stratify=True,
):
    """
    Train/Val/Test split with optional stratification.

    Requirements:
      train_frac + val_frac + test_frac == 1.0
    """
    total = train_frac + val_frac + test_frac
    assert abs(total - 1.0) < 1e-6, "train_frac + val_frac + test_frac must be 1.0"

    rng = np.random.default_rng(seed)
    n = len(y)

    if not stratify:
        idx = rng.permutation(n)
        n_train = int(n * train_frac)
        n_val = int(n * val_frac)

        train_idx = idx[:n_train]
        val_idx = idx[n_train : n_train + n_val]
        test_idx = idx[n_train + n_val :]
        return (
            (X[train_idx], y[train_idx]),
            (X[val_idx], y[val_idx]),
            (X[test_idx], y[test_idx]),
        )

    # Stratified: do per-class splitting
    classes, y_inv = np.unique(y, return_inverse=True)

    train_idx_all, val_idx_all, test_idx_all = [], [], []

    for c in range(len(classes)):
        cls_idx = np.where(y_inv == c)[0]
        cls_idx = rng.permutation(cls_idx)

        n_c = len(cls_idx)
        n_train_c = int(n_c * train_frac)
        n_val_c = int(n_c * val_frac)

        train_idx_all.append(cls_idx[:n_train_c])
        val_idx_all.append(cls_idx[n_train_c : n_train_c + n_val_c])
        test_idx_all.append(cls_idx[n_train_c + n_val_c :])

    train_idx = rng.permutation(np.concatenate(train_idx_all))
    val_idx = rng.permutation(np.concatenate(val_idx_all))
    test_idx = rng.permutation(np.concatenate(test_idx_all))

    return (
        (X[train_idx], y[train_idx]),
        (X[val_idx], y[val_idx]),
        (X[test_idx], y[test_idx]),
    )


(X_train, y_train), (X_val, y_val), (X_test, y_test) = make_splits_train_val_test(
    X_flat,
    y,
    train_frac=0.8,
    val_frac=0.1,
    test_frac=0.1,
    seed=42,
    stratify=True,
)

print(f"\nTraining samples:   {X_train.shape[0]}")
print(f"Validation samples: {X_val.shape[0]}")
print(f"Test samples:       {X_test.shape[0]}")
print(f"Features/sample:    {X_train.shape[1]}")


# --- 5) Sanity check label distribution ---
def label_hist(y_arr, name, topk=10):
    vals, cnts = np.unique(y_arr, return_counts=True)
    order = np.argsort(cnts)[::-1]
    print(f"\n{name} label distribution (top {topk}):")
    for v, c in zip(vals[order][:topk], cnts[order][:topk]):
        orig = idx_to_label[int(v)]
        print(f"  idx {int(v):2d} (CORINE {orig:2d}): {int(c)}")


label_hist(y_train, "Train")
label_hist(y_val, "Val")
label_hist(y_test, "Test")

Keys in npz: ['patches', 'labels']
Loaded X shape: (160000, 3, 3, 4), dtype=float32
Loaded y shape: (160000,), dtype=uint8
Flattened X shape: (160000, 36)

Label remapping (original -> index):
  CORINE  2 -> class index 0
  CORINE 12 -> class index 1
  CORINE 23 -> class index 2
  CORINE 24 -> class index 3
  CORINE 25 -> class index 4
  CORINE 29 -> class index 5
  CORINE 36 -> class index 6
  CORINE 41 -> class index 7
Total classes: 8

Training samples:   127996
Validation samples: 15997
Test samples:       16007
Features/sample:    36

Train label distribution (top 10):
  idx  3 (CORINE 24): 84716
  idx  1 (CORINE 12): 15904
  idx  5 (CORINE 29): 12122
  idx  7 (CORINE 41): 7808
  idx  6 (CORINE 36): 2726
  idx  0 (CORINE  2): 1833
  idx  4 (CORINE 25): 1726
  idx  2 (CORINE 23): 1161

Val label distribution (top 10):
  idx  3 (CORINE 24): 10589
  idx  1 (CORINE 12): 1988
  idx  5 (CORINE 29): 1515
  idx  7 (CORINE 41): 976
  idx  6 (CORINE 36): 340
  idx  0 (CORINE  2): 229
  idx 

### Class Imbalance & Sampling Strategies

Understanding the data distribution and picking a sampling strategy to handle class imbalance.

In [6]:
y_train_np = np.asarray(y_train, dtype=np.int64)
classes, counts = np.unique(y_train_np, return_counts=True)

print("=" * 70)
print("TRAINING SET CLASS DISTRIBUTION")
print("=" * 70)
print(f"Total samples: {len(y_train_np)}")
print(f"Number of classes: {len(classes)}\n")

# Sort by count (descending)
sort_idx = np.argsort(-counts)
print("Classes (sorted by frequency):")
print("Index | CORINE | Count  | Percentage | Balance")
print("-" * 50)
for idx in sort_idx:
    c, n = classes[idx], counts[idx]
    pct = 100 * n / len(y_train_np)
    bar = "█" * int(pct / 2)
    print(
        f"{int(c):5d} | {idx_to_label[int(c)]:6d} | {int(n):6d} | {pct:6.1f}%  | {bar}"
    )

majority_class_pct = 100 * counts.max() / len(y_train_np)
print(f"\n⚠️  Majority class represents {majority_class_pct:.1f}% of data")
print(f"    This is the baseline accuracy for a 'dumb' classifier!")

# Compute imbalance ratio
imbalance_ratio = counts.max() / counts.min()
print(f"    Imbalance ratio (max/min): {imbalance_ratio:.1f}x")

TRAINING SET CLASS DISTRIBUTION
Total samples: 127996
Number of classes: 8

Classes (sorted by frequency):
Index | CORINE | Count  | Percentage | Balance
--------------------------------------------------
    3 |     24 |  84716 |   66.2%  | █████████████████████████████████
    1 |     12 |  15904 |   12.4%  | ██████
    5 |     29 |  12122 |    9.5%  | ████
    7 |     41 |   7808 |    6.1%  | ███
    6 |     36 |   2726 |    2.1%  | █
    0 |      2 |   1833 |    1.4%  | 
    4 |     25 |   1726 |    1.3%  | 
    2 |     23 |   1161 |    0.9%  | 

⚠️  Majority class represents 66.2% of data
    This is the baseline accuracy for a 'dumb' classifier!
    Imbalance ratio (max/min): 73.0x


### Setting up the model

In [8]:
class ConvNet(pl.LightningModule):
    """
    Simple CNN classifier for square Sentinel-2 patches.

    Assumes flat input (B, h*w*in_channels) where spatial layout is
    channels-last: the flat was produced by X.reshape(N, -1) on an
    array of shape (N, h, w, in_channels).

    patch_size: spatial size (h = w), default 3 for 3x3 patches.
    in_channels: number of spectral bands (4 or 10).
    """

    def __init__(
        self,
        num_classes=10,
        in_channels=10,
        patch_size=3,
        lr=3e-4,
        max_epochs=60,
        class_weights=None,
    ):
        super().__init__()
        self.lr = lr
        self.in_channels = in_channels
        self.patch_size = patch_size
        self.num_classes = num_classes
        self.max_epochs = max_epochs

        # Feature extraction layers
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )

        # Classification head
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes),
        )

        if class_weights is not None:
            self.register_buffer("class_weights", class_weights.float())
        else:
            self.class_weights = None

    def _to_image(self, x):
        """
        Convert flat input to image format (B, C, H, W).
        Handles channels-last layout: (B, h*w*c) → (B, h, w, c) → (B, c, h, w)
        """
        x = x.float()
        h = w = self.patch_size
        c = self.in_channels
        if x.dim() == 2:
            assert (
                x.size(1) == h * w * c
            ), f"Expected {h * w * c} features, got {x.size(1)}"
            x = x.view(x.size(0), h, w, c).permute(0, 3, 1, 2).contiguous()
        return x

    def forward(self, x):
        x = self._to_image(x)
        x = self.features(x)
        return self.classifier(x)

    def _loss(self, logits, y):
        return F.cross_entropy(logits, y, weight=self.class_weights)

    def training_step(self, batch, _):
        x, y = batch
        y = y.long()
        logits = self(x)
        loss = self._loss(logits, y)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, _):
        x, y = batch
        y = y.long()
        logits = self(x)
        loss = self._loss(logits, y)
        acc = (logits.argmax(1) == y).float().mean()
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)
        return loss

    def test_step(self, batch, _):
        x, y = batch
        y = y.long()
        logits = self(x)
        loss = self._loss(logits, y)
        acc = (logits.argmax(1) == y).float().mean()
        self.log("test_loss", loss, prog_bar=True)
        self.log("test_acc", acc, prog_bar=True)
        return loss

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=self.max_epochs)
        return {"optimizer": opt, "lr_scheduler": sched}

In [9]:
# ============================================================================
# CONFIGURATION: Choose Your Sampling Strategy
# ============================================================================

batch_size = 256  # Reduced for better class mixing with sampling
max_epochs = 100

print("\n" + "=" * 70)
print("SAMPLING STRATEGY COMPARISON")
print("=" * 70)

# Strategy 1: Inverse-frequency weights for weighted loss
print("\n1️⃣  STRATEGY 1: Class-Weighted Loss (No Resampling)")
print("-" * 70)
print("   How it works:")
print("   • Each class gets a weight = 1 / frequency")
print("   • Model sees all samples in their original distribution")
print("   • Loss contribution per class is equalized")
print("   • Pros: Simple, no data duplication")
print("   • Cons: Model still biased toward majority class")

# Compute sqrt-scaled weights (softens extreme weights)
raw_weights = np.zeros(len(classes), dtype=np.float64)
for c, n in zip(classes, counts):
    raw_weights[int(c)] = 1.0 / n

sqrt_weights = np.sqrt(raw_weights)
sqrt_weights /= sqrt_weights.sum() / len(classes)  # Normalize to mean=1

print("\n   Class weights (sqrt-scaled):")
print("   Index | CORINE | Weight")
print("   " + "-" * 35)
sort_idx = np.argsort(-sqrt_weights)
for c_idx in sort_idx[: len(classes)]:
    orig = idx_to_label[c_idx]
    print(f"   {c_idx:5d} | {orig:6d} | {sqrt_weights[c_idx]:6.2f}")

class_weights_t = torch.tensor(sqrt_weights, dtype=torch.float32)

# Strategy 2: Weighted Random Sampler
print("\n\n2️⃣  STRATEGY 2: Weighted Random Sampler (Resampling)")
print("-" * 70)
print("   How it works:")
print("   • Each sample gets weight = 1 / class_frequency")
print("   • Sampler draws batches with replacement")
print("   • Rare classes appear more often in mini-batches")
print("   • Pros: Direct oversampling, minorities get more epochs")
print("   • Cons: Duplication, may overfit on rare classes")

# Map each sample to its weight
class_weight_by_value = {int(c): float(1.0 / n) for c, n in zip(classes, counts)}
sample_weights = np.array(
    [class_weight_by_value[int(lbl)] for lbl in y_train_np], dtype=np.float64
)

sampler = WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(sample_weights),
    replacement=True,
)

print("   ✓ Sampler created")

# ============================================================================
# SELECT WHICH STRATEGY TO USE
# ============================================================================
print("\n\n" + "=" * 70)
print("YOUR CHOICE: Which strategy will you use?")
print("=" * 70)

strategy = "weighted_sampler"  # ← CHANGE THIS TO "weighted_sampler" TO COMPARE

if strategy == "class_weights":
    print(f"\n✅ Selected: CLASS-WEIGHTED LOSS")
    class_weights_for_loss = class_weights_t
    train_sampler = None
    train_shuffle = True
    print(f"   • Using loss weights to balance classes")
    print(f"   • DataLoader will shuffle normally")

elif strategy == "weighted_sampler":
    print(f"\n✅ Selected: WEIGHTED RANDOM SAMPLER")
    class_weights_for_loss = None
    train_sampler = sampler
    train_shuffle = False
    print(f"   • Using resampling in DataLoader")
    print(f"   • Rare classes will appear more often")

else:
    print(f"\n✅ Selected: NO WEIGHTS")
    class_weights_for_loss = None
    train_sampler = None
    train_shuffle = True
    print(f"   • DataLoader will shuffle normally")

print("\n" + "=" * 70)
print("MODEL SETUP")
print("=" * 70)

# Auto-detect architecture parameters
num_classes = len(np.unique(y_train_np))
patch_size = 3  # 3x3 patches
in_channels = X_train.shape[1] // (patch_size**2)

print(f"\n  num_classes    = {num_classes}")
print(f"  patch_size     = {patch_size}×{patch_size}")
print(f"  in_channels    = {in_channels}")
print(f"  batch_size     = {batch_size}")

# Create model
model = ConvNet(
    lr=3e-4,
    num_classes=num_classes,
    in_channels=in_channels,
    patch_size=patch_size,
    class_weights=class_weights_for_loss,
    max_epochs=max_epochs,
)

print(f"\n✓ CNN Model created")

# Create datasets
train_dataset = YourCustomDataset(X_train, y_train)
val_dataset = YourCustomDataset(X_val, y_val)
test_dataset = YourCustomDataset(X_test, y_test)

# Create dataloaders
train_dataloader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    sampler=train_sampler,
    shuffle=train_shuffle,
    num_workers=2,
    pin_memory=True,
)

val_dataloader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

print(f"✓ DataLoaders created")
print(f"  • Training batches:   {len(train_dataloader)}")
print(f"  • Validation batches: {len(val_dataloader)}")
print(f"  • Test batches:       {len(test_dataloader)}")


SAMPLING STRATEGY COMPARISON

1️⃣  STRATEGY 1: Class-Weighted Loss (No Resampling)
----------------------------------------------------------------------
   How it works:
   • Each class gets a weight = 1 / frequency
   • Model sees all samples in their original distribution
   • Loss contribution per class is equalized
   • Pros: Simple, no data duplication
   • Cons: Model still biased toward majority class

   Class weights (sqrt-scaled):
   Index | CORINE | Weight
   -----------------------------------
       2 |     23 |   1.84
       4 |     25 |   1.51
       0 |      2 |   1.46
       6 |     36 |   1.20
       7 |     41 |   0.71
       5 |     29 |   0.57
       1 |     12 |   0.50
       3 |     24 |   0.22


2️⃣  STRATEGY 2: Weighted Random Sampler (Resampling)
----------------------------------------------------------------------
   How it works:
   • Each sample gets weight = 1 / class_frequency
   • Sampler draws batches with replacement
   • Rare classes appear more of